In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [1]:
# ---------- CELL 1: SETUP, IMPORTS, PATHS, GIT ----------
!pip install -U accelerate bitsandbytes peft transformers datasets rouge-score -q
print("Packages installed.")

import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import gc, random, json, yaml, time, numpy as np, pandas as pd, torch, torch.nn.functional as F
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, TrainerCallback, EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model
from datasets import Dataset as HFDataset
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print(f"\nGPUs: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)} - {torch.cuda.get_device_properties(i).total_memory / 1024**3:.1f} GB")

# ---------- PATHS ----------
MODEL_PATH = "/kaggle/input/datasets/tasfiahmrinmoyee/gemma3-1b-model/gemma-3-1b-it"
TRAIN_CSV = "/kaggle/input/datasets/tasfiahmrinmoyee/dataset-updated/train_resplit.csv"
VAL_CSV   = "/kaggle/input/datasets/tasfiahmrinmoyee/dataset-updated/val_resplit.csv"
TEST_CSV  = "/kaggle/input/datasets/tasfiahmrinmoyee/dataset-updated/test_resplit.csv"
OUTPUT_DIR = "/kaggle/working/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("\nChecking model path:")
!ls -lh {MODEL_PATH} | head -5

print("\nLoading CSVs...")
train_df = pd.read_csv(TRAIN_CSV); val_df = pd.read_csv(VAL_CSV); test_df = pd.read_csv(TEST_CSV)
for col in ['buggy_code', 'fixed_code']:
    train_df[col] = train_df[col].astype(str).str.strip()
    val_df[col] = val_df[col].astype(str).str.strip()
    test_df[col] = test_df[col].astype(str).str.strip()
print(f"Train: {len(train_df)}, Val: {len(val_df)}, Test: {len(test_df)}")

# ---------- GIT SETUP ----------
from kaggle_secrets import UserSecretsClient
GIT_USER_EMAIL = "tasfiahmrinmoyee@gmail.com"
GIT_USER_NAME = "Tasfiah Tasnim Mrinmoyee"
GITHUB_USERNAME = "tasfiahmrinmoyee"
REPO_NAME = "Gemma3-1B-Instruct-outputs"
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
if GITHUB_TOKEN is None:
    print("WARNING: GITHUB_TOKEN not found. Git push skipped.")
    GIT_REPO_PATH = None
else:
    REPO_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_USERNAME}/{REPO_NAME}.git"
    GIT_REPO_PATH = "/kaggle/working/outputs_repo"
    !git config --global user.email "$GIT_USER_EMAIL"
    !git config --global user.name "$GIT_USER_NAME"
    if not os.path.exists(GIT_REPO_PATH):
        !git clone $REPO_URL $GIT_REPO_PATH
    else:
        !cd $GIT_REPO_PATH && git pull
    # Setup LFS for checkpoints
    !cd $GIT_REPO_PATH && git lfs track "*.bin" "*.safetensors" "*.pt" "*.pth"
    !cd $GIT_REPO_PATH && git add .gitattributes
    !cd $GIT_REPO_PATH && git commit -m "Add LFS tracking" || echo "No changes"
    !cd $GIT_REPO_PATH && git push
    print("Git repository ready with LFS.")

print("\nCell 1 complete. Proceed to Cell 2.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 8.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 46.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 34.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 91.3 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 28.6 MB/s eta 0:00:00
Packages installed.

GPUs: 2
  GPU 0: Tesla T4 - 14.6 GB
  GPU 1: Tesla T4 - 14.6 GB

Checking model path:
total 1.9G
-rw-r--r-- 1 nobody nogroup   35 Aug  9 14:48 added_tokens.json
-rw-r--r-- 1 nobody nogroup  899 Aug  9 14:48 config.json
-rw-r--r-- 1 nobody nogroup  215 Aug  9 14:48 generation_config.json
-rw-r--r-- 1 nobody nogroup 1.9G Aug  9 14:48 model.safetensors

Loading CSVs...
Train: 20077, Val: 2510, Test: 2510
Cloning into '/kaggle/working/outputs_repo'...
Tracking "*.bin"
Tr

In [2]:
# ---------- CELL 2: LOAD MODEL + LORA ----------
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print(f"Pad: {tokenizer.pad_token}, EOS: {tokenizer.eos_token}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
)

print("Loading model (4-bit, device_map='auto')...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    dtype=torch.float16,
)
print("Model loaded.")

lora_config = LoraConfig(
    r=16,
    lora_alpha=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.0,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
model.gradient_checkpointing_enable()
print("Gradient checkpointing enabled.")

# Quick test
test_prompt = "### System: Fix buggy code.\n### User: x = 5\nprint(x)\n### Assistant:"
inputs = tokenizer(test_prompt, return_tensors="pt", truncation=True, max_length=64).to(model.device)
with torch.no_grad():
    outputs = model.generate(**inputs, max_new_tokens=10, do_sample=False)
print("Forward pass test:", tokenizer.decode(outputs[0], skip_special_tokens=True))

for i in range(torch.cuda.device_count()):
    print(f"GPU {i}: allocated {torch.cuda.memory_allocated(i)/1024**3:.2f} GB")
print("Cell 2 complete. Proceed to Cell 3.")

Loading tokenizer...
Pad: <pad>, EOS: <eos>
Loading model (4-bit, device_map='auto')...


Loading weights:   0%|          | 0/340 [00:00<?, ?it/s]

Model loaded.
trainable params: 13,045,760 || all params: 1,012,931,712 || trainable%: 1.2879
Gradient checkpointing enabled.
Forward pass test: ### System: Fix buggy code.
### User: x = 5
print(x)
### Assistant:
```python
x = 5
print
GPU 0: allocated 0.00 GB
GPU 1: allocated 0.95 GB
Cell 2 complete. Proceed to Cell 3.


In [3]:
# ---------- CELL 3: TOKENIZATION (max_len=768) ----------
from datasets import Dataset as HFDataset
import torch

print("Tokenizing with max_len=768...")

def format_prompt(buggy, fixed):
    return f"### System: Fix buggy code.\n### User: {buggy}\n### Assistant: {fixed}"

def tokenize_function(examples, tokenizer, max_len=768):
    texts = [format_prompt(b, f) for b, f in zip(examples['buggy_code'], examples['fixed_code'])]
    return tokenizer(texts, truncation=True, max_length=max_len, padding=False)

train_hf = HFDataset.from_pandas(train_df[['buggy_code','fixed_code']])
val_hf = HFDataset.from_pandas(val_df[['buggy_code','fixed_code']])

train_hf = train_hf.map(lambda x: tokenize_function(x, tokenizer), batched=True, remove_columns=['buggy_code','fixed_code'])
val_hf = val_hf.map(lambda x: tokenize_function(x, tokenizer), batched=True, remove_columns=['buggy_code','fixed_code'])
print(f"Train: {len(train_hf)}, Val: {len(val_hf)}")

class CustomDataCollator:
    def __init__(self, tokenizer):
        self.tokenizer = tokenizer
    def __call__(self, features):
        max_len = max(len(f['input_ids']) for f in features)
        pad_id = self.tokenizer.pad_token_id
        padded_input_ids = [f['input_ids'] + [pad_id]*(max_len-len(f['input_ids'])) for f in features]
        padded_attn = [f['attention_mask'] + [0]*(max_len-len(f['attention_mask'])) for f in features]
        padded_labels = [ids.copy() for ids in padded_input_ids]
        return {
            "input_ids": torch.tensor(padded_input_ids, dtype=torch.long),
            "attention_mask": torch.tensor(padded_attn, dtype=torch.long),
            "labels": torch.tensor(padded_labels, dtype=torch.long),
        }

custom_collator = CustomDataCollator(tokenizer)
print("Cell 3 complete. Proceed to Cell 4.")

Tokenizing with max_len=768...


Map:   0%|          | 0/20077 [00:00<?, ? examples/s]

Map:   0%|          | 0/2510 [00:00<?, ? examples/s]

Train: 20077, Val: 2510
Cell 3 complete. Proceed to Cell 4.


In [4]:
# ---------- CELL 4: TRAINING ARGS + COMBINED CALLBACK ----------
import json, os, shutil
from transformers import TrainerCallback, TrainingArguments, Trainer

print("Configuring training (save/eval every 200 steps, max_len=768)...")

TOTAL_STEPS = 3765
WARMUP_STEPS = int(0.03 * TOTAL_STEPS)   # 112

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,        # 2 GPUs × 2 × 4 = 16
    gradient_accumulation_steps=4,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    eval_strategy="steps",
    eval_steps=200,
    save_strategy="steps",
    save_steps=200,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    fp16=True,
    bf16=False,
    learning_rate=2e-4,
    warmup_steps=WARMUP_STEPS,
    weight_decay=0.01,
    seed=SEED,
    report_to="none",
    save_total_limit=2,
    dataloader_num_workers=2,
    gradient_checkpointing=True,
    ddp_find_unused_parameters=False,
)
print(f"Total steps: {TOTAL_STEPS}, Warmup: {WARMUP_STEPS}, Save/Eval: every 200 steps")

# ---------- COMBINED CALLBACK (Early Stopping + Git Push on Every Save) ----------
class CombinedCallback(TrainerCallback):
    def __init__(self, git_repo_path="/kaggle/working/outputs_repo"):
        self.git_repo_path = git_repo_path
        self.best_loss = float('inf')
        self.steps_without_improvement = 0
        self.patience_steps = 5   # 5 × 200 steps = 1000 steps without improvement

    def on_evaluate(self, args, state, control, **kwargs):
        # Early stopping logic
        eval_loss = None
        for log in reversed(state.log_history):
            if "eval_loss" in log:
                eval_loss = log["eval_loss"]
                break
        if eval_loss is None:
            return control
        print(f"\n=== Step {state.global_step}. Loss: {eval_loss:.4f} ===")
        with open(os.path.join(OUTPUT_DIR, f"val_loss_step_{state.global_step}.json"), 'w') as f:
            json.dump({"eval_loss": eval_loss, "step": state.global_step}, f)
        if eval_loss < self.best_loss:
            self.best_loss = eval_loss
            self.steps_without_improvement = 0
            print(f"New best loss: {eval_loss:.4f}")
        else:
            self.steps_without_improvement += 1
            if self.steps_without_improvement >= self.patience_steps:
                print("Early stopping triggered.")
                control.should_training_stop = True
        return control

    def on_save(self, args, state, control, **kwargs):
        # Push EVERY checkpoint (every 200 steps) to Git
        if self.git_repo_path and os.path.exists(self.git_repo_path):
            try:
                shutil.copytree(OUTPUT_DIR, self.git_repo_path, dirs_exist_ok=True)
                !cd {self.git_repo_path} && git add .
                !cd {self.git_repo_path} && git commit -m "Checkpoint at step {state.global_step}" || echo "No changes"
                !cd {self.git_repo_path} && git push
                print(f"Pushed checkpoint at step {state.global_step} to Git.")
            except Exception as e:
                print(f"Git push failed: {e}")
        return control

# ---------- INITIALIZE TRAINER ----------
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_hf,
    eval_dataset=val_hf,
    data_collator=custom_collator,
    callbacks=[CombinedCallback()]
)

print("Cell 4 complete. Proceed to Cell 5 (Start Training).")

Configuring training (save/eval every 200 steps, max_len=768)...
Total steps: 3765, Warmup: 112, Save/Eval: every 200 steps
Cell 4 complete. Proceed to Cell 5 (Start Training).


In [5]:
# ---------- CELL 5: START TRAINING ----------
print("\n=== STARTING TRAINING ===")
print(f"Total steps: {int(training_args.num_train_epochs * len(train_hf) / (training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps * torch.cuda.device_count()))}")
print("Checkpoint saved and pushed to Git every 200 steps.")
print("Early stopping patience: 5 evaluations (1000 steps) without loss improvement.")
print("Expected time: ~6–8 hours total (3 epochs).")

trainer.train()

# ---------- SAVE FINAL MODEL & LOGS ----------
model.save_pretrained(os.path.join(OUTPUT_DIR, "gemma_qlora_final"))
trainer.save_state()

with open(os.path.join(OUTPUT_DIR, "training_logs.json"), 'w') as f:
    json.dump(trainer.state.log_history, f)

# Update run_config with final stats
config_path = os.path.join(OUTPUT_DIR, "run_config.json")
if os.path.exists(config_path):
    with open(config_path, 'r') as f:
        final_config = json.load(f)
    final_config["completed_steps"] = trainer.state.global_step
    final_config["best_checkpoint"] = trainer.state.best_model_checkpoint
    final_config["best_metric"] = trainer.state.best_metric
    with open(config_path, 'w') as f:
        json.dump(final_config, f, indent=2)

# ---------- FINAL GIT PUSH ----------
if os.path.exists("/kaggle/working/outputs_repo"):
    try:
        import shutil
        shutil.copytree(OUTPUT_DIR, "/kaggle/working/outputs_repo", dirs_exist_ok=True)
        !cd /kaggle/working/outputs_repo && git add .
        !cd /kaggle/working/outputs_repo && git commit -m "Training completed: final model and logs" || echo "No new changes"
        !cd /kaggle/working/outputs_repo && git push
        print("Final outputs pushed to GitHub.")
    except Exception as e:
        print(f"Git push failed: {e}")

print("\n=== TRAINING COMPLETED ===")
print(f"Best model checkpoint: {trainer.state.best_model_checkpoint}")
print(f"Best eval loss: {trainer.state.best_metric:.4f}")
print(f"Final model saved to: {OUTPUT_DIR}/gemma_qlora_final")
print("All outputs (checkpoints, logs, config) saved to:", OUTPUT_DIR)


=== STARTING TRAINING ===
Total steps: 3764
Checkpoint saved and pushed to Git every 200 steps.
Early stopping patience: 5 evaluations (1000 steps) without loss improvement.
Expected time: ~6–8 hours total (3 epochs).


Step,Training Loss,Validation Loss
200,0.889721,0.997348
400,0.701319,0.913688
600,0.658480,0.870645
800,0.617219,0.834514
1000,0.541546,0.802635
1200,0.533565,0.784737
1400,0.488708,0.773037
1600,0.524761,0.764823
1800,0.432843,0.755837
2000,0.466108,0.751652



=== Step 200. Loss: 0.9973 ===
New best loss: 0.9973
[main 805730b] Checkpoint at step 200
 13 files changed, 2380034 insertions(+)
 create mode 100644 checkpoint-200/README.md
 create mode 100644 checkpoint-200/adapter_config.json
 create mode 100644 checkpoint-200/adapter_model.safetensors
 create mode 100644 checkpoint-200/chat_template.jinja
 create mode 100644 checkpoint-200/optimizer.pt
 create mode 100644 checkpoint-200/rng_state.pth
 create mode 100644 checkpoint-200/scaler.pt
 create mode 100644 checkpoint-200/scheduler.pt
 create mode 100644 checkpoint-200/tokenizer.json
 create mode 100644 checkpoint-200/tokenizer_config.json
 create mode 100644 checkpoint-200/trainer_state.json
 create mode 100644 checkpoint-200/training_args.bin
 create mode 100644 val_loss_step_200.json
Enumerating objects: 17, done./6), 157 MB | 68 MB/s                             
Counting objects: 100% (17/17), done.
Delta compression using up to 4 threads
Compressing objects: 100% (15/15), done.
Writ